In [ ]:
!pip install transformers accelerate bitsandbytes datasets peft sentencepiece scikit-learn -q

from huggingface_hub import login
login()

In [ ]:
!pip install -U transformers -q


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model

model_name = "meta-llama/Llama-2-7b-chat-hf"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=True)

# Add padding token (CRITICAL)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': tokenizer.eos_token})

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    device_map="auto",
    load_in_4bit=True,
    torch_dtype="auto"
)

# Sync padding with model
model.config.pad_token_id = tokenizer.pad_token_id
model.resize_token_embeddings(len(tokenizer))


/usr/local/lib/python3.12/dist-packages/transformers/models/auto/tokenization_auto.py:1041: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Llama-2-7b-chat-hf and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Embedding(32000, 4096)

In [ ]:
config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    task_type="SEQ_CLS"
)

model = get_peft_model(model, config)
model.print_trainable_parameters()

trainable params: 8,400,896 || all params: 6,615,756,800 || trainable%: 0.1270


In [ ]:
import pandas as pd

train_df = pd.read_csv("train_shortened.csv")

min_count = train_df["label"].value_counts().min()

target_per_class = 300

train_df = (
    train_df
    .groupby("label")
    .sample(n=target_per_class, random_state=42)
    .reset_index(drop=True)
)

print(train_df.shape)
print(train_df["label"].value_counts())

(900, 2)
label
0    300
1    300
2    300
Name: count, dtype: int64


In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict

train_df = pd.read_csv("train_shortened.csv")
val_df   = pd.read_parquet("validation-00000-of-00001.parquet")
test_df  = pd.read_parquet("test-00000-of-00001.parquet")

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df),
})

print(dataset["train"].column_names)

['sentence', 'label']


In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["sentence"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

tokenized = dataset.map(tokenize, batched=True)

# Remove unused columns safely
cols_to_remove = [c for c in tokenized["train"].column_names if c not in ["input_ids", "attention_mask", "label"]]
tokenized = tokenized.remove_columns(cols_to_remove)
tokenized = tokenized.rename_column("label", "labels")

Map:   0%|          | 0/1449 [00:00<?, ? examples/s]

Map:   0%|          | 0/484 [00:00<?, ? examples/s]

Map:   0%|          | 0/484 [00:00<?, ? examples/s]

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro")
    }

In [ ]:
from transformers import TrainingArguments

In [ ]:

args = TrainingArguments(
    output_dir="llama2-finance",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=1e-4,
    logging_steps=25,
    bf16=True,
    optim="paged_adamw_32bit",
    report_to=[],
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics,
)


In [ ]:
from datasets import Dataset, DatasetDict

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df),
})

In [ ]:
trainer.train()

results = trainer.evaluate()
print("Evaluation Results:", results)

Step,Training Loss
25,12.340700
50,10.295000
75,7.374900
100,5.631100
125,5.440300
150,3.325500
175,4.250800
200,2.624300
225,2.726500
250,2.922800


Evaluation Results: {'eval_loss': 0.778175950050354, 'eval_accuracy': 0.8367768595041323, 'eval_f1': 0.8312314539709589, 'eval_runtime': 870.9258, 'eval_samples_per_second': 0.556, 'eval_steps_per_second': 0.556, 'epoch': 3.0}
